In [ ]:
# ======================================================================
# LEVEL 7 PRIVATE EDGE (NO ORACLE / NO CREDIT CARD) - Qwen2.5 on Colab T4
# ======================================================================
# Ladder:  groq (public)  >  THIS Colab edge (private, T4)  >  phone 1.5B
#
# STEPS:
#   1) Colab  ->  Runtime  ->  Change runtime type  ->  T4 GPU
#   2) Run cells in order.
#   3) Paste your FREE ngrok authtoken when asked (no credit card).
#   4) Copy the https://....ngrok-free.app printed, set in Vercel:
#        JARVIS_EDGE_URL   = https://....ngrok-free.app
#        JARVIS_EDGE_AUTH  = <same token you paste here>
#   5) Redeploy; health should flip oracle_edge 'down' -> 'up'.
#
# NOTE: a Colab runtime stops after hours idle / daily cap. Acceptable
#       for dev; re-run to bring it back.
# ======================================================================


In [ ]:
# Cell 1 - install Ollama and pull the Q4 model (fits T4 16GB)
!curl -fsSL https://ollama.com/install.sh | sh
!ollama serve &> /tmp/ollama.log &
!sleep 4
!ollama pull mariojnick/qwen2.5:7b-instruct-q4_k_m
# if the community tag is missing, try:   !ollama pull qwen2.5:7b-instruct


In [ ]:
# Cell 2 - confirm ollama is up and model is cached
import subprocess, time
print(subprocess.run(['ollama','list'],capture_output=True,text=True).stdout)


In [ ]:
# Cell 3 - expose via ngrok (public HTTPS, free, NO credit card)
!pip -q install pyngrok >/dev/null
import os, json
from pyngrok import ngrok
os.environ['NGROK_AUTHTOKEN'] = input('Paste your FREE ngrok authtoken: ').strip()
tunnel = ngrok.connect(11434, proto='http')
public_url = tunnel.public_url.strip()
print('JARVIS_EDGE_URL =', public_url)


In [ ]:
# Cell 4 - smoke test the public OpenAI-compatible endpoint
import urllib.request, json
print('public /health ->', urllib.request.urlopen(public_url + '/health', timeout=25).status)
req = urllib.request.Request(
    public_url + '/v1/chat/completions',
    data=json.dumps({
        'model': 'mariojnick/qwen2.5:7b-instruct-q4_k_m',
        'messages': [{'role':'user','content':'Sebutkan satu ibukota Indonesia dalam satu kata.'}],
        'stream': False,
    }).encode(),
    headers={'Content-Type':'application/json'},
  )
with urllib.request.urlopen(req, timeout=180) as r:
    d = json.loads(r.read())
print('reply:', d['choices'][0]['message']['content'][:200])
